# DeepLabCut 工具箱 - 开放场景演示 (Open-Field DEMO)

以下是一些可能有用的资源：

- [github.com/DeepLabCut/DeepLabCut](https://github.com/DeepLabCut/DeepLabCut)
- [DeepLabCut 文档：单动物项目用户指南](https://deeplabcut.github.io/DeepLabCut/docs/standardDeepLabCut_UserGuide.html)

#### 本 Notebook 对应以下用户指南：

Nath\*, Mathis\* 等人，《*Using DeepLabCut for markerless pose estimation during behavior across species*》，自然方案（Nature Protocols），2019 年：https://www.nature.com/articles/s41596-019-0176-0

本 Notebook 演示了如何进行以下操作：
- 加载演示项目
- 训练网络
- 评估网络
- 分析新视频
- 创建自动标记的视频
- 绘制轨迹
- 识别异常帧
- 手动标注异常帧
- 合并数据集并更新训练集
- 训练网络

注意：本 Notebook 从一个已经初始化并包含已标记数据的项目开始。

这些数据是下列论文的一个子集：《*DeepLabCut: markerless pose estimation of user-defined body parts with deep learning*》https://www.nature.com/articles/s41593-018-0209-y（此子集未用于训练我们论文中展示或评估的模型）。

In [ ]:
# Importing the toolbox (takes several seconds)
from pathlib import Path

import deeplabcut

In [ ]:
# Create a variable to set the config.yaml file path:
# If this path does not point to the project from the URL below,
# edit it to make sure it does:
#   https://github.com/DeepLabCut/DeepLabCut/tree/main/examples/openfield-Pranav-2018-10-30
# 
# Example - Linux/OSX
#   path_config_file = "/Users/john/DeepLabCut/examples/openfield-Pranav-2018-10-30/config.yaml"
# Example - Windows
#   path_config_file = r"C:\DeepLabCut\examples\openfield-Pranav-2018-10-30\config.yaml"
#
# Note that parameters of this project can be seen at: *openfield-Pranav-2018-10-30/config.yaml*

path_config_file = str(Path.cwd() / "openfield-Pranav-2018-10-30" / "config.yaml")
deeplabcut.load_demo_data(path_config_file)

In [ ]:
# [OPTIONAL] Perhaps plot the labels to see how the frames were annotated:
# (note, this project was created in Linux, so you might have an error in Windows, but this is an optional step)

deeplabcut.check_labels(path_config_file)

## 开始训练特征检测器（Feature Detectors）

此函数针对训练数据集的特定洗牌（shuffle）来训练网络。用户可以在 `/openfield-Pranav-2018-10-30/dlc-models-pytorch/.../pytorch_config.yaml` 文件中设置各种参数。有关可设置变量的更多信息，请查阅 [文档](https://deeplabcut.github.io/DeepLabCut/docs/pytorch/pytorch_config.html)！

训练可以随时停止。请注意，权重仅在每经过 'save\_epochs' 个周期（epoch）时才会被保存。对于此演示，建议非常频繁地保存和显示进度。但在实际应用中，这样做效率很低。您应该会在大约 50 到 60 个周期后看到模型开始收敛；您可以继续训练更长时间以提高性能。

In [ ]:
# notice the variables "save_epochs" and "displayiters" that can be set in the function
deeplabcut.train_network(
    path_config_file,
    shuffle=1,
    save_epochs=2,
    displayiters=5,
)

**请注意，如果程序运行至结束或您手动停止它（通过点击“停止”或按 CTRL+C），您会看到一个 "KeyboardInterrupt" 错误，但您可以忽略此错误！**

## 评估已训练的网络

此函数用于评估一个已训练的模型，针对特定的洗牌（shuffle/shuffles），或在特定的训练状态（快照/snapshot）下进行评估，也可以评估所有状态。评估是在数据集（图像）上进行的，并将结果以 `.csv` 文件的形式存储在 `evaluation-results-pytorch` 目录下的一个子目录中。

您可以在该项目的 `config.yaml` 文件中更改各种参数。对于评估来说，模型的所有描述符（如 Task、TrainingFraction、Date 等）都非常重要。在评估过程中，可以更改 `pcutoff` 参数。此截止值（cutoff）也会影响到在绘图中显示的估计位置所需的概率阈值。此外，还可以更改这些图表的颜色映射（colormap）和点的大小（dotsize）。

In [ ]:
deeplabcut.evaluate_network(path_config_file, plotting=False)

*注意：根据您的设置，有时您可能会遇到一些 "matplotlib 错误，但这些错误并不重要*

现在您可以去查看这些图片了。考虑到数据输入有限，并且这次测试花费了大约 20 分钟，因此它不会产生很好的跟踪效果，所以不必为此感到惊讶。这只是为了让您熟悉这个工作流程……

## 分析视频

该函数从视频中提取基于训练网络的姿态信息。用户可以选择所使用的训练网络——默认情况下，会使用最新快照（snapshot）来分析视频。然而，用户也可以在 `config.yaml` 文件中为变量 `snapshotindex` 指定快照的索引。

分析结果将存储在与视频位于**同一目录下的 hd5 文件**中。姿态数组（即姿态与帧索引的关系）也可以导出为 CSV 文件（设置标志位...）。

In [ ]:
videofile_path = str(Path(path_config_file).parent / "videos" / "m3v1mp4.mp4")

In [ ]:
print("Start analyzing the video!")
# our demo video on a CPU with take ~5 min to analze! GPU is much faster!
deeplabcut.analyze_videos(path_config_file, [videofile_path])

## 创建带标签的视频

此函数用于可视化目的，可以用来创建一个带有预测标签的 `.mp4` 格式视频。此视频将保存在与（未标记的）原始视频相同的目录下。

可以针对颜色映射（colormap）和点的大小（dotsize）设置各种参数。这些参数的...

In [ ]:
deeplabcut.create_labeled_video(path_config_file, [videofile_path])

## 绘制分析视频的轨迹

此函数会绘制整个视频中所有身体部位的轨迹。每个身体部位都会由一个唯一的颜色来标识。底层的函数可以轻松进行自定义。

In [ ]:
%matplotlib notebook
deeplabcut.plot_trajectories(
    path_config_file,
    [videofile_path],
    showfigures=True,
)

# These plots are interactive and can be customized (see https://matplotlib.org/)

## 提取预测结果存在偏差的异常帧

这是一个可选步骤，当评估结果不佳时，可以用来增加更多的训练数据。在这种情况下，用户可以使用以下函数来提取那些标签被错误预测的帧。请确保提供正确的 `iterations` 值，因为它将用于创建保存提取出的帧的唯一目录。

In [ ]:
deeplabcut.extract_outlier_frames(path_config_file, [videofile_path])

用户可以**迭代地**运行此操作，甚至可以从**同一视频中提取额外的帧**。

## 手动校正标签

此步骤允许用户校正提取出的帧中的标签。导航到对应视频 'm3v1mp4' 的文件夹，并按照协议中所述使用 GUI 来更新这些标签。

有关 GUI 的文档，请[参阅 `napari-deeplabcut` 的文档](https://github.com/DeepLabCut/napari-deeplabcut/tree/main)——特别是关于 _“3. 精炼标签——图像文件夹包含一个 machinelabels-iter\<#>.h5 文件”_ 的部分！

In [ ]:
deeplabcut.refine_labels(path_config_file)

In [ ]:
# Perhaps plot the labels to see how how all the frames are annotated (including the refined ones)
deeplabcut.check_labels(path_config_file)

In [ ]:
# Now merge datasets (once you refined all frames)
deeplabcut.merge_datasets(path_config_file)

## 创建新的训练数据集迭代、检查并训练...

遵循完善后的标签，将这些帧追加到原始数据集中，以创建新的训练数据集迭代。

In [ ]:
deeplabcut.create_training_dataset(path_config_file)

现在，我们可以再次训练网络了（使用扩展后的数据集）。我们可以通过使用 `snapshot_path` 参数来从已有的快照继续训练——与从头开始训练模型不同，它会加载我们已有的权重并对其进行微调！

In [ ]:
snapshot_path = (  # Edit me if needed! Select the path to the snapshot to continue training from!
    Path(path_config_file).parent / 
    "dlc-models-pytorch" / 
    "iteration-0" / 
    "openfieldOct30-trainset95shuffle1" / 
    "train" / 
    "snapshot-best-080.pt"
)

deeplabcut.train_network(
    path_config_file,
    shuffle=1,
    save_epochs=2,
    displayiters=10,
    batch_size=8,
    snapshot_path=snapshot_path,
)